In [5]:
from ..structure_output import *
#结构化输出,四种模式里只有pydantic会对响应做校验
#pydantic
load_dotenv(override=True)

DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')
#必须关闭思考模式，deepseek默认为思考模式，思考模式不支持结构化输出
#因为deepseek底层使用tool_choice作为伪工具传递结构化输出，思考模式不支持tool_choice
model=init_chat_model(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model='deepseek-v4-flash',
    model_provider='deepseek',
    extra_body={"thinking":{"type":"disabled"}}
)
#结构化输出
class movieModel(BaseModel):
    movie_name: str=Field(
        description='电影名'
    )
    movie_director: str=Field(
        description='导演'
    )
    movie_score: float=Field(
        description='电影评分'
    )
    movie_brief: str=Field(
        description='电影简介'
    )

movie_model=model.with_structured_output(movieModel)
movie=movie_model.invoke('2012是什么电影')
print(type(movie))
print(movie.movie_name)
print(movie.movie_director)
print(movie.movie_score)
print(movie.movie_brief)


<class '__main__.movieModel'>
2012
罗兰·艾默里奇
8.0
《2012》是一部2009年上映的美国科幻灾难片，由罗兰·艾默里奇执导。影片根据玛雅预言中2012年12月21日世界末日的传说展开，讲述了全球各地发生地震、火山爆发、海啸等巨大自然灾害，人类面临灭绝危机，少数人乘坐诺亚方舟逃生的故事。


In [6]:
#默认值
class product(BaseModel):
    name:str=Field(
        description='产品名称'
    )
    price:float=Field(
        description='产品价格'
    )
    des:str=Field(
        description='产品描述',
        default='暂无描述'
    )
    stock:int=Field(
        description='产品库存',
        default=0
    )
product_model=model.with_structured_output(product)
prod=product_model.invoke('iqqo12 售价2999 一款不错的手机 库存30台')
print(type(prod))
print(prod.name)
print(prod.price)
print(prod.des)
print(prod.stock)

<class '__main__.product'>
iqqo12
2999.0
一款不错的手机
30


In [7]:
#也可以使用Literal
#可选字段和枚举类型
class Priority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"
class CustomerInfo(BaseModel):
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")
customer_model=model.with_structured_output(CustomerInfo)
conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""
customerInfo = customer_model.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(customerInfo)

name='王小明' phone='138-1234-5678' email='null' issue='订单一直没发货' urgency=<Priority.HIGH: '高'>
